In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from useful_func import missing_info,out_info

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [ ]:
# load data
df = pd.read_csv("/content/h1n1_vaccine_prediction.csv")
df.head()

,unique_id,h1n1_worry,h1n1_awareness,antiviral_medication,contact_avoidance,bought_face_mask,wash_hands_frequently,avoid_large_gatherings,reduced_outside_home_cont,avoid_touch_face,...,race,sex,income_level,marital_status,housing_status,employment,census_msa,no_of_adults,no_of_children,h1n1_vaccine
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,White,Female,Below Poverty,Not Married,Own,Not in Labor Force,Non-MSA,0.0,0.0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,White,Male,Below Poverty,Not Married,Rent,Employed,"MSA, Not Principle City",0.0,0.0,0
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,White,Male,"<= $75,000, Above Poverty",Not Married,Own,Employed,"MSA, Not Principle City",2.0,0.0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,White,Female,Below Poverty,Not Married,Rent,Not in Labor Force,"MSA, Principle City",0.0,0.0,0
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,White,Female,"<= $75,000, Above Poverty",Married,Own,Employed,"MSA, Not Principle City",1.0,0.0,0


In [ ]:
# split data X and Y
df.columns
X = df.iloc[:, 1:-1]
y = df['h1n1_vaccine']

In [ ]:
# check data is balanced or not
y.value_counts(1)

,proportion
h1n1_vaccine,
0,0.787546
1,0.212454


In [ ]:
# do train test
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    stratify = y,
                                                    random_state = 42)
print(f"shape of X_train -:{X_train.shape}")
print(f"shape of X_test -:{X_test.shape}")
print(f"shape of y_train -:{y_train.shape}")
print(f"shape of y_test -:{y_test.shape}")

shape of X_train -:(20030, 32)
shape of X_test -:(6677, 32)
shape of y_train -:(20030,)
shape of y_test -:(6677,)


In [ ]:
# data cleaning
## for has_health_insur impute with 0 and add missing_indicator
health_insur_imp = SimpleImputer(strategy = "constant", fill_value = 0,
                               add_indicator = True)
health_insur_imp.fit(X_train[["has_health_insur"]])
col_names = health_insur_imp.get_feature_names_out()
X_train[col_names] = health_insur_imp.transform(X_train[["has_health_insur"]])
X_test[col_names] = health_insur_imp.transform(X_test[["has_health_insur"]])

In [ ]:
X_train[col_names]

,has_health_insur,missingindicator_has_health_insur
11075,1.0,0.0
7807,0.0,1.0
3014,1.0,0.0
1671,0.0,1.0
16691,0.0,1.0
...,...,...
17823,1.0,0.0
10210,1.0,0.0
7737,0.0,1.0
12227,1.0,0.0


In [ ]:
# income_level --> mode with missing_indicator
# -moderate_missing_cols --> mode with missing_indicator

moderate_missing_cols = ['income_level','dr_recc_h1n1_vacc',
                         'dr_recc_seasonal_vacc', 'housing_status',
                        'employment', 'marital_status', 'qualification']
moderate_missing_col_imp = SimpleImputer(strategy = "most_frequent",
                               add_indicator = True)
moderate_missing_col_imp.fit(X_train[moderate_missing_cols])
col_names = moderate_missing_col_imp.get_feature_names_out()
X_train[col_names] = (moderate_missing_col_imp.
                     transform(X_train[moderate_missing_cols]))
X_test[col_names] = (moderate_missing_col_imp.
                    transform(X_test[moderate_missing_cols]))




In [ ]:
low_missing_cols = ['chronic_medic_condition', 'cont_child_undr_6_mnths',
       'is_health_worker', 'sick_from_seas_vacc', 'is_seas_risky',
       'is_seas_vacc_effective'] + ['sick_from_h1n1_vacc', 'is_h1n1_vacc_effective', 'is_h1n1_risky',
       'no_of_adults', 'no_of_children', 'contact_avoidance',
       'avoid_touch_face', 'h1n1_awareness', 'h1n1_worry',
       'avoid_large_gatherings', 'reduced_outside_home_cont',
       'antiviral_medication', 'wash_hands_frequently',
       'bought_face_mask']

In [ ]:
low_missing_cols = ['income_level','dr_recc_h1n1_vacc',
                         'dr_recc_seasonal_vacc', 'housing_status',
                        'employment', 'marital_status', 'qualification']
low_missing_col_imp = SimpleImputer(strategy = "most_frequent")
low_missing_col_imp.fit(X_train[low_missing_cols])
col_names = low_missing_col_imp.get_feature_names_out()
X_train[col_names] = (low_missing_col_imp.
                     transform(X_train[low_missing_cols]))
X_test[col_names] = (low_missing_col_imp.
                    transform(X_test[low_missing_cols]))

In [ ]:
missing_info(X_test)

,column_name,num_missing,missing%
0,chronic_medic_condition,228,3.41
1,cont_child_undr_6_mnths,188,2.82
2,is_health_worker,184,2.76
3,sick_from_seas_vacc,120,1.80
4,is_seas_risky,120,1.80
5,is_seas_vacc_effective,100,1.50
6,is_h1n1_vacc_effective,91,1.36
7,is_h1n1_risky,87,1.30
8,sick_from_h1n1_vacc,84,1.26
9,no_of_adults,64,0.96


In [ ]:
# orinal encoding
ordinal = ["age_bracket", "qualification", "income_level"]
for col in ordinal:
  print(col)
print(X_train[col].unique())
print("-----------------------------")




age_bracket
qualification
income_level
['<= $75,000, Above Poverty' '> $75,000' 'Below Poverty']
-----------------------------


In [ ]:
age_order = ['18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years']
qualification_order = ['< 12 Years', '12 Years', 'Some College', 'College Graduate']
income_order = ['Below Poverty', '<= $75,000, Above Poverty', '> $75,000']

In [ ]:
ord_enc = OrdinalEncoder(categories=[age_order, qualification_order,
                                     income_order])
X_train[ordinal]= ord_enc. fit_transform(X_train[ordinal])
X_test[ordinal] = ord_enc. transform(X_test[ordinal])

In [ ]:
X_train[ordinal]

,age_bracket,qualification,income_level
11075,1.0,2.0,1.0
7807,1.0,3.0,1.0
3014,4.0,1.0,1.0
1671,0.0,1.0,1.0
16691,4.0,3.0,2.0
...,...,...,...
17823,0.0,1.0,0.0
10210,4.0,1.0,0.0
7737,4.0,2.0,1.0
12227,1.0,3.0,2.0


In [ ]:
nominal = ["census_msa", "employment", "housing_status",
"marital_status","sex", "race"]
ohe_enc = OneHotEncoder (sparse_output=False, drop = "first")
ohe_enc.fit(X_train[nominal])
col_names = ohe_enc.get_feature_names_out()
X_train[col_names] = ohe_enc.transform(X_train[nominal])
X_test[col_names] = ohe_enc.transform(X_test[nominal])

In [ ]:
X_train.to_csv("X_train.csv")
y_train.to_csv("y_train.csv")
X_test.to_csv("X_test.csv")
y_test.to_csv("y_test.csv")

NameError: name 'X_train' is not defined